# LC 739 — Daily Temperatures
**Difficulty:** Medium | **Category:** Monotonic Stack
**Pattern:** Decreasing Monotonic Stack (Next Greater Element)

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Keep a stack of indices whose temperatures
haven't found a warmer day yet. The stack stays in decreasing order
of temperature. When a warmer day arrives, pop and record the gap.
</div>

## Official Problem Statement

Given an array of integers `temperatures` representing daily temperatures,
return an array `answer` such that `answer[i]` is the number of days you
have to wait after the `i`-th day to get a warmer temperature.
If there is no future day with a warmer temperature, keep `answer[i] == 0`.

**Constraints:**
- `1 <= temperatures.length <= 100_000`
- `30 <= temperatures[i] <= 100`

## What This Is Actually Asking

You have a list of temperatures, one per day.
For each day, how many days until it gets warmer?
If it never gets warmer, the answer is 0.
You're finding the "next greater element" distance for each position.
Return a list of those waiting times.

## Walk Through an Example by Hand

```
temperatures = [73, 74, 75, 71, 69, 72, 76, 73]
indices        [ 0,  1,  2,  3,  4,  5,  6,  7]

stack=[] answer=[0,0,0,0,0,0,0,0]

i=0, temp=73: stack empty → push 0.  stack=[0]
i=1, temp=74: 74>temps[0]=73 → pop 0, ans[0]=1-0=1
              stack empty → push 1.  stack=[1]
i=2, temp=75: 75>temps[1]=74 → pop 1, ans[1]=2-1=1
              stack empty → push 2.  stack=[2]
i=3, temp=71: 71<temps[2]=75 → push 3.  stack=[2,3]
i=4, temp=69: 69<temps[3]=71 → push 4.  stack=[2,3,4]
i=5, temp=72: 72>temps[4]=69 → pop 4, ans[4]=5-4=1
              72>temps[3]=71 → pop 3, ans[3]=5-3=2
              72<temps[2]=75 → push 5.  stack=[2,5]
i=6, temp=76: 76>temps[5]=72 → pop 5, ans[5]=6-5=1
              76>temps[2]=75 → pop 2, ans[2]=6-2=4
              stack empty → push 6.  stack=[6]
i=7, temp=73: 73<temps[6]=76 → push 7.  stack=[6,7]

Result: [1,1,4,2,1,1,0,0]
```

## The Picture

```
Temp bar chart (height = temperature - 68):

  76 |              ██
  75 |        ██    ██
  74 |     ██ ██    ██
  73 |  ██ ██ ██    ██ ██
  72 |  ██ ██ ██    ██ ██ ██
  71 |  ██ ██ ██ ██ ██ ██ ██
  69 |  ██ ██ ██ ██ ██ ██ ██
       [0] [1] [2] [3] [4] [5] [6] [7]
        73  74  75  71  69  72  76  73

Stack at each step (stores indices, shown as temps):
  After i=2: [75]          ← waiting for warmer
  After i=4: [75, 71, 69]  ← all waiting
  After i=5: [75, 72]      ← 71,69 resolved
  After i=6: [76]          ← 75,72 resolved
  After i=7: [76, 73]      ← still waiting → ans=0

Key: stack is always decreasing (top = coldest waiting day)
```

## When To Use This Pattern

- When you see "next greater/smaller element to the right", think
  monotonic stack.
- When you need the *distance* to the next greater element, think
  storing indices (not values) in the stack.
- When elements need to "wait" until some future condition is met,
  think a stack of unresolved indices.
- When brute force is O(n²) with a nested "look ahead" loop, think
  monotonic stack for O(n).
- When the stack must stay sorted to be useful, think monotonic.

## The Approach

Initialize a result array of zeros and an empty stack.
Iterate through each index: while the stack is not empty and the
current temperature beats the temperature at the stack's top index,
pop that index and record the gap (current index minus popped index).
Push the current index onto the stack.
Any indices left in the stack at the end stay 0 (no warmer day found).

In [6]:
from typing import List  # type hints for function signatures

In [7]:
def test_harness(func):
    """Run test cases for dailyTemperatures."""
    tests = [
        # (input_temperatures, expected_output)
        ([73,74,75,71,69,72,76,73], [1,1,4,2,1,1,0,0]),
        ([30,40,50,60],             [1,1,1,0]),
        ([30,60,90],                [1,1,0]),
        ([90,80,70,60],             [0,0,0,0]),
        ([50],                      [0]),
    ]
    passed = 0
    for i, (temps, expected) in enumerate(tests):
        result = func(temps)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(f"Test {i+1}: {status}")
        if status == "FAILED":
            print(f"  Input:    {temps}")
            print(f"  Expected: {expected}")
            print(f"  Got:      {result}")
    print(f"\n{passed}/{len(tests)} tests passed.")

In [8]:
def dailyTemperatures(temps: List[int]) -> List[int]:
    """
    For each day, return how many days until a warmer temperature.

    Args:
        temperatures: list of daily temperatures
    Returns:
        list of wait times (0 if no warmer day exists)

    Approach:
        Decreasing monotonic stack of indices.
        Pop when current temp > temp at stack top.
        Record gap = current_index - popped_index.
    """
    # Initialize a zero list for every day.
    # Initialize an empty stack to house indices of monotonic decreasing temperatures.
    # For a temperature warmer than the stack top, evict the index and populate the result.
    # Example: daily_temperatures([90, 60, 30]) -> [0, 0, 0] (a decreasing monotonic stack).
    stack = []
    res = [0] * len(temps)
    for i, temp in enumerate(temps):
        while stack and temp > temps[stack[-1]]:
            idx = stack.pop()
            res[idx] = i - idx
        stack.append(i)
    return res



# --- Debug prints (expected in comments) ---
t1 = [73, 74, 75, 71, 69, 72, 76, 73]
print(dailyTemperatures(t1))  # [1,1,4,2,1,1,0,0]

t2 = [30, 40, 50, 60]
print(dailyTemperatures(t2))  # [1,1,1,0]

t3 = [90, 80, 70, 60]
print(dailyTemperatures(t3))  # [0,0,0,0]

t4 = [50]
print(dailyTemperatures(t4))  # [0]
test_harness(dailyTemperatures)

[1, 1, 4, 2, 1, 1, 0, 0]
[1, 1, 1, 0]
[0, 0, 0, 0]
[0]
Test 1: PASSED
Test 2: PASSED
Test 3: PASSED
Test 4: PASSED
Test 5: PASSED

5/5 tests passed.


In [ ]:
# Uncomment and run when solution is ready
# test_harness(dailyTemperatures)

## Complexity

| Approach        | Time   | Space  |
|-----------------|--------|--------|
| Brute Force     | O(n²)  | O(1)   |
| Monotonic Stack | O(n)   | O(n)   |

Each index is pushed and popped at most once → O(n) total.

## Real World Connection

At Citi, telemetry from 6,000 endpoints streams CPU metrics every minute.
A common alert pattern is: "how many minutes until CPU drops below
this spike?" — the inverse of Daily Temperatures.
AWS CloudWatch anomaly detection uses exactly this kind of "next
crossing" logic to set alarm recovery times.
A monotonic stack processes the stream in O(n) rather than rescanning
history for every new data point, which matters at scale.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra